In [2]:
# === Unsupervised extras (memory-safe): t-SNE (sample), Hierarchical (sample), metrics, silhouettes, network ===
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage

from project_package.modeling import (
    run_kmeans_from_csv, run_isolation_forest_from_csv, run_pca_embeddings_from_csv,
    load_csv_dedup, EXCLUDE_ALWAYS, _unsup_pre_ordinal
)

# ---------------- Config (tune if needed) ----------------
TSNE_SAMPLE_MAX   = 20_000   # t-SNE only for visualization
HIER_SAMPLE_MAX   = 5_000    # dendrogram must use a sample (hierarchical is O(n^2))
K_LIST            = [3, 5, 8, 10]
RANDOM_STATE      = 42
IDS               = ["Booking ID", "Customer ID"]

# ---------------- Paths ----------------
ROOT = Path.cwd()
CSV  = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists(): CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

OUT = ROOT / "unsupervised"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- Core unsupervised models (full data) ----------------
km_res = run_kmeans_from_csv(str(CSV), id_cols=IDS, artifacts_dir=str(OUT), k_list=K_LIST, random_state=RANDOM_STATE)
iso    = run_isolation_forest_from_csv(str(CSV), id_cols=IDS, artifacts_dir=str(OUT), contamination=0.02, random_state=RANDOM_STATE)
emb    = run_pca_embeddings_from_csv(str(CSV), id_cols=IDS, artifacts_dir=str(OUT), n_components=2, random_state=RANDOM_STATE)

print("Core saved:")
print("  KMeans:", km_res.model_path, "| labels:", km_res.labels_csv_path, "| pca2:", km_res.pca2_csv_path)
print("  IsoForest:", iso.model_path, "| scores:", iso.scores_csv_path)
print("  PCA 2D:", emb.embed_csv_path)

# ---------------- Build X exactly like modeling.py (full data) ----------------
df = load_csv_dedup(str(CSV))
drop_cols = set(EXCLUDE_ALWAYS) | set(IDS)
X_raw = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
pre = _unsup_pre_ordinal(X_raw)
X  = pre.fit_transform(X_raw).astype(np.float32, copy=False)  # cast to float32 to save memory
n  = X.shape[0]
print(f"[INFO] Feature matrix: shape={X.shape}, dtype={X.dtype}")

# Keep IDs (if present)
id_frame = pd.DataFrame({c: df[c].values for c in IDS if c in df.columns})

# Helper: stratified subsample indices (by KMeans labels if available), else random
def stratified_sample_idx(labels: np.ndarray | None, max_n: int, rng: np.random.Generator):
    if max_n >= n:
        return np.arange(n)
    if labels is None:
        return rng.choice(n, size=max_n, replace=False)
    idx_per = []
    # proportional sample per cluster
    _, counts = np.unique(labels, return_counts=True)
    for k_val in np.unique(labels):
        k_idx = np.where(labels == k_val)[0]
        take = max(1, int(round(len(k_idx) / n * max_n)))
        take = min(take, len(k_idx))
        idx_per.append(rng.choice(k_idx, size=take, replace=False))
    sel = np.concatenate(idx_per)
    if len(sel) > max_n:  # trim if rounding overflow
        sel = sel[:max_n]
    return sel

rng = np.random.default_rng(RANDOM_STATE)

# Read back best-k labels from file (so we can stratify samples)
km_labels_df = pd.read_csv(km_res.labels_csv_path)
km_labels = km_labels_df["cluster"].values if "cluster" in km_labels_df.columns else None

# ---------------- KMeans metrics across K (full data) ----------------
metrics_rows = []
best_k, best_sil = km_res.best_k, -np.inf
best_km, best_labels = None, None

for k in K_LIST:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    sil = float(silhouette_score(X, labels))
    dbi = float(davies_bouldin_score(X, labels))
    ch  = float(calinski_harabasz_score(X, labels))
    centroids = km.cluster_centers_
    wcss = float(((X - centroids[labels])**2).sum())
    tss  = float(((X - X.mean(axis=0))**2).sum())
    bss  = tss - wcss
    wb_ratio = float(wcss / bss) if bss > 0 else np.inf
    metrics_rows.append({"k":k, "silhouette":sil, "dbi":dbi, "ch":ch, "wcss":wcss, "bss":bss, "w_over_b":wb_ratio})
    if sil > best_sil:
        best_sil, best_k, best_km, best_labels = sil, k, km, labels

metrics_df = pd.DataFrame(metrics_rows)
metrics_path = OUT / "kmeans_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("[OK] kmeans_metrics.csv ->", metrics_path, "| best_k:", best_k, "sil:", round(best_sil,4))

# ---------------- Per-sample silhouette (best KMeans, full data) ----------------
sil_vals = silhouette_samples(X, best_labels)
sil_df = pd.DataFrame({"cluster": best_labels, "silhouette": sil_vals})
sil_df = pd.concat([sil_df, id_frame], axis=1)
sil_path = OUT / f"silhouette_samples_k{best_k}.csv"
sil_df.to_csv(sil_path, index=False)
print("[OK] silhouette_samples ->", sil_path)

# ---------------- t-SNE on stratified sample ----------------
tsne_idx = stratified_sample_idx(km_labels, TSNE_SAMPLE_MAX, rng)
X_tsne = X[tsne_idx]
tsne_ids = id_frame.iloc[tsne_idx] if not id_frame.empty else None
print(f"[INFO] t-SNE sample size: {len(tsne_idx)} / {n}")

tsne2 = TSNE(n_components=2, init="pca", perplexity=30, learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X_tsne)
tsne_df = pd.DataFrame({"tsne1": tsne2[:,0], "tsne2": tsne2[:,1]})
if tsne_ids is not None: tsne_df = pd.concat([tsne_df, tsne_ids.reset_index(drop=True)], axis=1)
tsne_path = OUT / "tsne_2d.csv"
tsne_df.to_csv(tsne_path, index=False)
print("[OK] tsne_2d.csv ->", tsne_path)

# ---------------- Network plot: centroid nodes (PCA2) + edges ----------------
pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
pca_df = pd.DataFrame({"pc1": pca2[:,0], "pc2": pca2[:,1], "cluster": best_labels})
centroids_pca = pca_df.groupby("cluster")[["pc1","pc2"]].mean().reset_index().rename(columns={"cluster":"node"})
centroids_pca_path = OUT / "kmeans_centroids_pca2.csv"
centroids_pca.to_csv(centroids_pca_path, index=False)

edges = []
nodes = centroids_pca["node"].tolist()
coords = centroids_pca.set_index("node")[["pc1","pc2"]].to_dict("index")
for i in range(len(nodes)):
    for j in range(i+1, len(nodes)):
        a, b = nodes[i], nodes[j]
        dx = coords[a]["pc1"] - coords[b]["pc1"]
        dy = coords[a]["pc2"] - coords[b]["pc2"]
        dist = float(np.sqrt(dx*dx + dy*dy))
        edges.append({"source": int(a), "target": int(b), "distance": dist})
edges_df = pd.DataFrame(edges)
edges_path = OUT / "kmeans_centroid_edges.csv"
edges_df.to_csv(edges_path, index=False)
print("[OK] network nodes/edges ->", centroids_pca_path, edges_path)

# ---------------- Hierarchical (sample only) ----------------
hier_idx = stratified_sample_idx(km_labels, HIER_SAMPLE_MAX, rng)
X_hier = X[hier_idx]
hier_ids = id_frame.iloc[hier_idx] if not id_frame.empty else None
print(f"[INFO] Hierarchical sample size: {len(hier_idx)} / {n}")

# Ward linkage (sample) -> dendrogram input
Z = linkage(X_hier, method="ward")  # O(m^2), m = sample size
Z_df = pd.DataFrame(Z, columns=["idx1","idx2","distance","sample_count"])
Z_path = OUT / "hierarchical_linkage_ward_sample.csv"
Z_df.to_csv(Z_path, index=False)
print("[OK] hierarchical_linkage_ward_sample.csv ->", Z_path)

# Flat labels for common k on the SAME sample (for colored dendrogram cuts)
for k in K_LIST:
    agg = AgglomerativeClustering(n_clusters=k, metric="euclidean", linkage="ward")
    h_labels = agg.fit_predict(X_hier)
    h_df = pd.DataFrame({"cluster": h_labels})
    if hier_ids is not None: h_df = pd.concat([h_df, hier_ids.reset_index(drop=True)], axis=1)
    out_k = OUT / f"hierarchical_labels_k{k}_sample.csv"
    h_df.to_csv(out_k, index=False)
    print(f"[OK] hierarchical_labels_k{k}_sample.csv ->", out_k)

print("\n[Done] All unsupervised artifacts saved to:", OUT.resolve())


Core saved:
  KMeans: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5.pkl | labels: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_labels.csv | pca2: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_pca2.csv
  IsoForest: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\isoforest.pkl | scores: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\isoforest_scores.csv
  PCA 2D: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\pca_2d.csv
[INFO] Feature matrix: shape=(146614, 41), dtype=float32
[OK] kmeans_metrics.csv -> c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_metrics.csv | best_k: 5 sil: 0.3565
[OK] silhouette_samples -> c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\silhouette_samples_k5.csv
[INFO] t-SNE sample size: 20000 / 146614
[OK] tsne_2d.csv -> c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\tsne_2d.csv
[OK] ne